# Spark SQL functions — a tour of all 27 function groups

One section per group of the [Spark SQL built-in functions reference](https://spark.apache.org/docs/latest/api/sql/), in the order of the reference:
a short description, **every function of the group** (names exactly as in the reference) and a **PySpark** cell with runnable examples.

**How to read the example cells.** Most cells call `show_examples([...])`: every item is a piece of PySpark code (as text) that is evaluated
on a one-row DataFrame and shown as **`pyspark` → `result` → `type`**, one line per function. Functions that have no
`pyspark.sql.functions` wrapper (yet) are called through `F.expr("<SQL>")` — exactly the SQL from the reference.
Aggregate, window and generator functions change the number of rows, so their cells show ordinary DataFrames instead.

Legend: 🆕 recent addition (the Spark version is given where the reference states it) · 🟠 Databricks specific.

In [0]:
# Setup — imports, one helper and two small sample DataFrames used by every section
import datetime as dt
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

def _fmt(v):
    if v is None:
        return "NULL"
    if isinstance(v, (bytes, bytearray)):
        return "0x" + bytes(v).hex().upper()
    return str(v)

def show_examples(codes, df=None):
    """Evaluate every PySpark expression (given as code text) on ONE row of df (default: a dummy row)
    and show one line per expression: the code, its result and its Spark SQL type."""
    df = spark.range(1) if df is None else df.limit(1)
    res = df.select(*[eval(code, globals()).alias(f"_{i}") for i, code in enumerate(codes)])
    row, types = res.first(), [f.dataType.simpleString() for f in res.schema.fields]
    display(spark.createDataFrame([(c, _fmt(row[i]), types[i]) for i, c in enumerate(codes)],
                                  "pyspark string, result string, type string"))

emp = spark.createDataFrame([
    (1, "Alice", "Board",       300000, 50000.0, dt.date(2015, 1, 10), dt.datetime(2026, 11, 2, 9, 15),  ["exec", "founder"], [9, 8, 10], {"level": "C",  "site": "WRO"}, "alice@acme.io"),
    (2, "Bob",   "Engineering", 200000, None,    dt.date(2016, 3, 1),  dt.datetime(2026, 11, 3, 8, 5),   ["dev", "lead"],     [7, 8],     {"level": "L6", "site": "WAW"}, "bob@acme.io"),
    (3, "Carol", "Sales",       190000, 15000.0, dt.date(2016, 7, 15), dt.datetime(2026, 11, 3, 17, 40), ["sales"],           [9],        {"level": "M2"},                "carol@acme.io"),
    (4, "Dan",   "Engineering", 150000, 5000.0,  dt.date(2018, 2, 1),  dt.datetime(2026, 11, 10, 7, 55), ["dev"],             [6, 7, 7],  {"level": "L4", "site": "WRO"}, "dan@acme.io"),
    (5, "Eve",   "Sales",       120000, None,    dt.date(2019, 9, 1),  dt.datetime(2026, 11, 12, 12, 0), ["sales", "intern"], [5],        {"level": "M1"},                None),
    (6, "Frank", "Engineering", 150000, 2000.0,  dt.date(2021, 5, 17), None,                             [],                  [],         {},                             "frank@acme.io"),
], "emp_id int, name string, dept string, salary int, bonus double, hired date, last_login timestamp, "
   "tags array<string>, scores array<int>, props map<string,string>, email string")
emp.createOrReplaceTempView("emp")

one = spark.range(1).drop("id")          # an empty one-row DataFrame for constant expressions
display(emp)

## 1. Agg Functions (86)

Aggregate functions collapse many rows into one value per group (`groupBy(...).agg(...)`, or the whole table with `agg(...)`).
Besides the classics (`count`, `sum`, `avg`, `min`, `max`) the group has **"by" aggregates** (`max_by` / `min_by` return *another* column
of the row with the extreme value), **ordered-set aggregates** (`percentile_cont`, `listagg … WITHIN GROUP`), statistics and
regression (`corr`, `covar_*`, `regr_*`, `skewness`, `kurtosis`), **ANSI-safe** `try_sum` / `try_avg`, and **sketch builders**
(`hll_*`, `theta_*`, `kll_*`, `tuple_*`, `approx_top_k_*`, `count_min_sketch`) whose results are read by the Sketch functions (section 18).

**Functions:** `any` · `any_value` · `approx_count_distinct` · `approx_percentile` · `approx_top_k` · `approx_top_k_accumulate` · `approx_top_k_combine` · `array_agg` · `avg` · `bit_and` · `bit_or` · `bit_xor` · `bitmap_and_agg` · `bitmap_construct_agg` · `bitmap_or_agg` · `bool_and` · `bool_or` · `collect_list` · `collect_set` · `collect_union` · `corr` · `count` · `count_if` · `count_min_sketch` · `covar_pop` · `covar_samp` · `every` · `first` · `first_value` · `grouping` · `grouping_id` · `histogram_numeric` · `hll_sketch_agg` · `hll_union_agg` · `kll_merge_agg_bigint` · `kll_merge_agg_double` · `kll_merge_agg_float` · `kll_sketch_agg_bigint` · `kll_sketch_agg_double` · `kll_sketch_agg_float` · `kurtosis` · `last` · `last_value` · `listagg` · `max` · `max_by` · `mean` · `measure` · `median` · `min` · `min_by` · `mode` · `percentile` · `percentile_approx` · `percentile_cont` · `percentile_disc` · `regr_avgx` · `regr_avgy` · `regr_count` · `regr_intercept` · `regr_r2` · `regr_slope` · `regr_sxx` · `regr_sxy` · `regr_syy` · `skewness` · `some` · `std` · `stddev` · `stddev_pop` · `stddev_samp` · `string_agg` · `sum` · `theta_intersection_agg` · `theta_sketch_agg` · `theta_union_agg` · `try_avg` · `try_sum` · `tuple_intersection_agg_double` · `tuple_intersection_agg_integer` · `tuple_sketch_agg_double` · `tuple_sketch_agg_integer` · `tuple_union_agg_double` · `tuple_union_agg_integer` · `var_pop` · `var_samp` · `variance`

Notes: `first` / `last` / `any_value` are non-deterministic without an ordering · `count(col)` skips NULLs, `count(*)` does not ·
`approx_*` trade exactness for memory · `grouping` / `grouping_id` tell which columns a ROLLUP / CUBE row aggregated away.
[Reference](https://spark.apache.org/docs/latest/api/sql/agg-functions/)

In [0]:
# 1a. The everyday aggregates, per department
display(emp.groupBy("dept").agg(
    F.count("*").alias("count"),
    F.count("bonus").alias("count_bonus"),                         # NULLs are not counted
    F.count_if(F.col("salary") >= 150000).alias("count_if_150k"),
    F.sum("salary").alias("sum"),
    F.avg("salary").alias("avg"),
    F.median("salary").alias("median"),
    F.min("salary").alias("min"),
    F.max("salary").alias("max"),
    F.max_by("name", "salary").alias("max_by_name"),               # name of the best-paid person
    F.min_by("name", "salary").alias("min_by_name"),
    F.sort_array(F.collect_list("name")).alias("collect_list"),
    F.sort_array(F.collect_set("salary")).alias("collect_set"),
    F.expr("listagg(name, ', ') WITHIN GROUP (ORDER BY name)").alias("listagg"),
    F.expr("string_agg(name, '|') WITHIN GROUP (ORDER BY name)").alias("string_agg"),
    F.first("bonus", ignorenulls=True).alias("first_non_null_bonus"),
    F.bool_and(F.col("salary") > 100000).alias("bool_and"),
    F.bool_or(F.col("bonus").isNull()).alias("bool_or"),
    F.bit_or("emp_id").alias("bit_or"),
    F.try_sum("salary").alias("try_sum"),
).orderBy("dept"))

# 1b. Statistics, percentiles, regression and approximate / sketch aggregates over the whole table
display(emp.agg(
    F.stddev("salary").alias("stddev"),
    F.variance("salary").alias("variance"),
    F.skewness("salary").alias("skewness"),
    F.kurtosis("salary").alias("kurtosis"),
    F.mode("salary").alias("mode"),
    F.percentile("salary", F.lit(0.9)).alias("p90_exact"),
    F.percentile_approx("salary", [0.25, 0.5, 0.75]).alias("quartiles_approx"),
    F.expr("percentile_cont(0.5) WITHIN GROUP (ORDER BY salary)").alias("percentile_cont"),
    F.expr("percentile_disc(0.5) WITHIN GROUP (ORDER BY salary)").alias("percentile_disc"),
    F.corr("salary", "bonus").alias("corr"),
    F.covar_pop("salary", "bonus").alias("covar_pop"),
    F.regr_slope("bonus", "salary").alias("regr_slope"),
    F.regr_count("bonus", "salary").alias("regr_count"),
    F.histogram_numeric("salary", F.lit(3)).alias("histogram_numeric"),
    F.approx_count_distinct("dept").alias("approx_count_distinct"),
    F.expr("hll_sketch_estimate(hll_sketch_agg(dept))").alias("hll_distinct_depts"),
    F.expr("bitmap_count(bitmap_construct_agg(bitmap_bit_position(emp_id)))").alias("bitmap_count"),
))

# 1c. grouping() / grouping_id(): which rows of a ROLLUP are subtotals
display(emp.rollup("dept").agg(F.grouping("dept").alias("grouping"), F.grouping_id().alias("grouping_id"),
                               F.sum("salary").alias("total")).orderBy("grouping", "dept"))

## 2. Array Functions (25)

Build, search, combine and reshape `ARRAY` values: set operations (`array_union`, `array_intersect`, `array_except`, `arrays_overlap`),
editing (`array_append`, `array_prepend`, `array_insert`, `array_remove`, `array_compact` drops NULLs), lookup (`array_contains`,
`array_position` is 1-based, `get` is 0-based and NULL-safe) and generation (`sequence`, `array_repeat`).

**Functions:** `array` · `array_append` · `array_compact` · `array_contains` · `array_distinct` · `array_except` · `array_insert` · `array_intersect` · `array_join` · `array_max` · `array_min` · `array_position` · `array_prepend` · `array_remove` · `array_repeat` · `array_size` · `array_union` · `arrays_overlap` · `arrays_zip` · `flatten` · `get` · `sequence` · `shuffle` · `slice` · `sort_array`

Higher-order functions (`transform`, `filter`, `aggregate`, …) and `element_at` / `size` live in **Collection functions** (section 5).
[Reference](https://spark.apache.org/docs/latest/api/sql/array-functions/)

In [0]:
arr = spark.createDataFrame([([3, 1, 2, 3, None], [2, 4])], "a array<int>, b array<int>")

show_examples([
    "F.array(F.lit(1), F.lit(2), F.lit(3))",
    "F.array_append('a', F.lit(9))",
    "F.array_prepend('a', F.lit(0))",
    "F.array_insert('a', 2, F.lit(99))",            # 1-based position
    "F.array_compact('a')",                         # drops NULLs
    "F.array_contains('a', 2)",
    "F.array_distinct('a')",
    "F.array_except('a', 'b')",
    "F.array_intersect('a', 'b')",
    "F.array_union('a', 'b')",
    "F.arrays_overlap('a', 'b')",
    "F.array_join('a', '-', 'NULL')",
    "F.array_max('a')",
    "F.array_min('a')",
    "F.array_position('a', 3)",                     # 1-based, 0 = not found
    "F.array_remove('a', 3)",
    "F.array_repeat(F.lit('x'), 3)",
    "F.array_size('a')",
    "F.arrays_zip('a', 'b')",
    "F.flatten(F.array('a', 'b'))",
    "F.get('a', 0)",                                # 0-based, NULL when out of range
    "F.sequence(F.lit(1), F.lit(10), F.lit(3))",
    "F.shuffle('b')",                               # random order
    "F.slice('a', 2, 3)",
    "F.sort_array('a', asc=False)",
], arr)

## 3. Avro Functions (3)

Serialize a struct to **Avro binary** (`to_avro`) and back (`from_avro`), e.g. for Kafka payloads; `schema_of_avro` shows the
Spark DDL type an Avro JSON schema maps to. `from_avro` needs the writer's Avro schema as JSON.

**Functions:** `from_avro` · `schema_of_avro` · `to_avro`

Note: the specified schema must match the data being read — otherwise the result is undefined. [Reference](https://spark.apache.org/docs/latest/api/sql/avro-functions/)

In [0]:
from pyspark.sql.avro.functions import to_avro, from_avro

# to_avro without a schema derives one: a record "topLevelRecord" whose nullable fields are ["<type>", "null"] unions
avro_schema = ('{"type": "record", "name": "topLevelRecord", "fields": ['
               '{"name": "emp_id", "type": ["int", "null"]}, {"name": "name", "type": ["string", "null"]}]}')

avro = emp.select("emp_id", to_avro(F.struct("emp_id", "name")).alias("avro_bytes"))
display(avro.select("emp_id", F.hex("avro_bytes").alias("avro_hex"),
                    from_avro("avro_bytes", avro_schema).alias("decoded")))

show_examples(["F.expr(f\"schema_of_avro('{avro_schema}', map())\")"])

## 4. Bitwise Functions (13)

Bit-level operators on integral types: `&` AND, `|` OR, `^` XOR, `~` NOT, shifts `<<`, `>>` (arithmetic) and `>>>` (unsigned),
plus `bit_count` (number of 1 bits) and `bit_get` / `getbit` (one bit, position 0 = least significant).
In PySpark the operators are `Column.bitwiseAND / bitwiseOR / bitwiseXOR`, `F.bitwise_not` and `F.shiftleft / shiftright / shiftrightunsigned`.

**Functions:** `&` · `<<` · `>>` · `>>>` · `^` · `bit_count` · `bit_get` · `getbit` · `shiftleft` · `shiftright` · `shiftrightunsigned` · `|` · `~`

[Reference](https://spark.apache.org/docs/latest/api/sql/bitwise-functions/)

In [0]:
bits = spark.createDataFrame([(12, 10)], "a int, b int")      # 12 = 1100, 10 = 1010

show_examples([
    "F.col('a').bitwiseAND(F.col('b'))",      "F.expr('a & b')",
    "F.col('a').bitwiseOR(F.col('b'))",       "F.expr('a | b')",
    "F.col('a').bitwiseXOR(F.col('b'))",      "F.expr('a ^ b')",
    "F.bitwise_not('a')",                     "F.expr('~a')",
    "F.shiftleft('a', 2)",                    "F.expr('a << 2')",
    "F.shiftright('a', 2)",                   "F.expr('a >> 2')",
    "F.shiftrightunsigned(F.lit(-12), 28)",   "F.expr('-12 >>> 28')",
    "F.bin('a')",
    "F.bit_count('a')",
    "F.bit_get('a', F.lit(2))",
    "F.getbit('a', F.lit(0))",
], bits)

## 5. Collection Functions (18)

Functions that work on **arrays and maps alike** (`size` / `cardinality`, `element_at`, `concat`, `reverse`) and the
**higher-order functions** that take a lambda: `transform`, `filter`, `exists`, `forall`, `aggregate` / `reduce`, `zip_with`,
`array_sort` with a comparator, and their map versions `transform_keys`, `transform_values`, `map_filter`, `map_zip_with`.
In PySpark the lambda is a Python function over Columns: `F.transform("a", lambda x: x * 10)`.

**Functions:** `aggregate` · `array_sort` · `cardinality` · `concat` · `element_at` · `exists` · `filter` · `forall` · `map_filter` · `map_zip_with` · `reduce` · `reverse` · `size` · `transform` · `transform_keys` · `transform_values` · `try_element_at` · `zip_with`

Notes: `element_at` is **1-based** and accepts negative indexes (from the end); under ANSI mode an out-of-range index raises an error,
`try_element_at` returns NULL instead. [Reference](https://spark.apache.org/docs/latest/api/sql/collection-functions/)

In [0]:
coll = spark.createDataFrame([([3, 1, 2], [10, 20, 30], {"x": 1, "y": 2}, {"x": 10, "z": 30})],
                             "a array<int>, b array<int>, m map<string,int>, n map<string,int>")

show_examples([
    "F.transform('a', lambda x: x * 10)",
    "F.transform('a', lambda x, i: x + i)",                      # (element, index)
    "F.filter('a', lambda x: x > 1)",
    "F.exists('a', lambda x: x > 2)",
    "F.forall('a', lambda x: x > 0)",
    "F.aggregate('a', F.lit(0), lambda acc, x: acc + x)",
    "F.reduce('a', F.lit(1), lambda acc, x: acc * x)",
    "F.zip_with('a', 'b', lambda x, y: x + y)",
    "F.array_sort('a')",
    "F.array_sort('a', lambda l, r: F.when(l < r, 1).when(l > r, -1).otherwise(0))",   # descending
    "F.size('a')",
    "F.cardinality('m')",
    "F.concat('a', 'b')",
    "F.element_at('a', -1)",                                      # last element
    "F.element_at('m', F.lit('y'))",
    "F.try_element_at('a', F.lit(10))",
    "F.reverse('a')",
    "F.transform_keys('m', lambda k, v: F.upper(k))",
    "F.transform_values('m', lambda k, v: v * 100)",
    "F.map_filter('m', lambda k, v: v > 1)",
    "F.map_zip_with('m', 'n', lambda k, v1, v2: F.coalesce(v1, F.lit(0)) + F.coalesce(v2, F.lit(0)))",
], coll)

## 6. Conditional Functions (12)

Choose a value: `CASE` / `when` (in PySpark `F.when(...).otherwise(...)`), `if`, the NULL helpers `coalesce`, `ifnull` / `nvl`, `nvl2`,
`nullif`, 🆕 `nullifzero` / `zeroifnull`, `nanvl` for floating-point NaN, and the range test `between`.

**Functions:** `between` · `case` · `coalesce` · `if` · `ifnull` · `nanvl` · `nullif` · `nullifzero` · `nvl` · `nvl2` · `when` · `zeroifnull`

Note: NULL and NaN are different things — `coalesce` does not replace NaN, `nanvl` does. [Reference](https://spark.apache.org/docs/latest/api/sql/conditional-functions/)

In [0]:
cond = spark.createDataFrame([(None, 0, 5, float("nan"))], "x int, y int, z int, d double")

show_examples([
    "F.coalesce('x', 'z')",
    "F.ifnull('x', F.lit(-1))",
    "F.nvl('x', F.lit(-1))",
    "F.nvl2('x', F.lit('has value'), F.lit('is NULL'))",
    "F.nullif('y', F.lit(0))",
    "F.expr('nullifzero(y)')",
    "F.expr('zeroifnull(x)')",
    "F.nanvl('d', F.lit(0.0))",
    "F.coalesce('d', F.lit(0.0))",                                    # NaN is not NULL
    "F.when(F.col('z') > 3, 'big').when(F.col('z') > 0, 'small').otherwise('none')",
    "F.expr(\"CASE WHEN z > 3 THEN 'big' ELSE 'small' END\")",
    "F.expr(\"CASE z WHEN 5 THEN 'five' ELSE 'other' END\")",          # simple CASE
    "F.expr(\"if(z > 3, 'big', 'small')\")",
    "F.col('z').between(1, 10)",
    "F.expr('z BETWEEN 1 AND 10')",
], cond)

## 7. Conversion Functions (14)

Type casts. `cast(expr AS type)` is the general form (PySpark `Column.cast("type")`); the other functions are **shorthands named after
the target type** (`int('42')`, `date('2026-11-18')`, …). 🆕 Spark 4.0 adds the `expr::type` cast operator, and `try_cast`
returns NULL instead of failing under ANSI mode. `time` builds the 🆕 `TIME` type.

**Functions:** `bigint` · `binary` · `boolean` · `cast` · `date` · `decimal` · `double` · `float` · `int` · `smallint` · `string` · `time` · `timestamp` · `tinyint`

[Reference](https://spark.apache.org/docs/latest/api/sql/conversion-functions/)

In [0]:
conv = spark.createDataFrame([("42", "3.14159", "2026-11-18 13:45:30", "abc")], "i string, d string, ts string, bad string")

show_examples([
    "F.col('i').cast('int')",
    "F.expr('int(i)')",
    "F.expr('bigint(i)')",
    "F.expr('smallint(i)')",
    "F.expr('tinyint(i)')",
    "F.col('d').cast('decimal(10,2)')",
    "F.expr('decimal(d)')",
    "F.expr('double(d)')",
    "F.expr('float(d)')",
    "F.expr(\"boolean('true')\")",
    "F.expr('date(ts)')",
    "F.expr('timestamp(ts)')",
    "F.expr(\"typeof(time('13:45:30'))\")",
    "F.expr(\"string(time('13:45:30'))\")",
    "F.expr('string(42)')",
    "F.expr(\"binary('Spark')\")",
    "F.expr('i::int + 1')",                        # Spark 4.0 cast operator
    "F.col('bad').try_cast('int')",                # NULL instead of an error
], conv)

## 8. Csv Functions (3)

Parse a CSV **string column** into a struct (`from_csv`, schema in DDL form), infer that schema from a sample (`schema_of_csv`) and
format a struct as one CSV line (`to_csv`). Options are the CSV data-source options (`sep`, `timestampFormat`, …).

**Functions:** `from_csv` · `schema_of_csv` · `to_csv`

All three exist since Spark 3.0. [Reference](https://spark.apache.org/docs/latest/api/sql/csv-functions/)

In [0]:
show_examples([
    "F.from_csv(F.lit('1, 0.8'), 'a INT, b DOUBLE')",
    "F.from_csv(F.lit('26/08/2015'), 'time TIMESTAMP', {'timestampFormat': 'dd/MM/yyyy'})",
    "F.schema_of_csv(F.lit('1,abc'))",
    "F.to_csv(F.struct(F.lit(1).alias('a'), F.lit(2).alias('b')))",
])

# Round trip on a table: struct -> CSV line (with ';' as separator) -> struct again
csv = emp.select("emp_id", F.to_csv(F.struct("emp_id", "name", "salary"), {"sep": ";"}).alias("csv_line"))
display(csv.withColumn("parsed", F.from_csv("csv_line", "emp_id INT, name STRING, salary INT", {"sep": ";"})))

## 9. Datetime Functions (82)

Everything about `DATE`, `TIMESTAMP` (an instant, shown in the session time zone), `TIMESTAMP_NTZ` (wall-clock, no zone), intervals and the
🆕 `TIME` type (Spark 4.1: `make_time`, `time_trunc`, `time_diff`, `to_time`, …). Families: current values, arithmetic
(`date_add`, `add_months`, `datediff`, `months_between`), truncation (`date_trunc`, `trunc`), field extraction (`year` … `second`,
`extract`, `date_part`), formatting / parsing (`date_format`, `to_date`, `to_timestamp`, `try_to_*`), epoch conversions
(`unix_*`, `timestamp_seconds`, …), time zones (`from_utc_timestamp`, `convert_timezone`) and event-time **windows** (`window`,
`session_window`, `window_time`).

**Functions:** `add_months` · `convert_timezone` · `curdate` · `current_date` · `current_time` · `current_timestamp` · `current_timezone` · `date_add` · `date_diff` · `date_format` · `date_from_unix_date` · `date_part` · `date_sub` · `date_trunc` · `dateadd` · `datediff` · `datepart` · `day` · `dayname` · `dayofmonth` · `dayofweek` · `dayofyear` · `extract` · `from_unixtime` · `from_utc_timestamp` · `hour` · `last_day` · `localtime` · `localtimestamp` · `make_date` · `make_dt_interval` · `make_interval` · `make_time` · `make_timestamp` · `make_timestamp_ltz` · `make_timestamp_ntz` · `make_ym_interval` · `minute` · `month` · `monthname` · `months_between` · `next_day` · `now` · `quarter` · `second` · `session_window` · `time_bucket` · `time_diff` · `time_from_micros` · `time_from_millis` · `time_from_seconds` · `time_to_micros` · `time_to_millis` · `time_to_seconds` · `time_trunc` · `timestamp_micros` · `timestamp_millis` · `timestamp_nanos` · `timestamp_seconds` · `to_date` · `to_time` · `to_timestamp` · `to_timestamp_ltz` · `to_timestamp_ntz` · `to_unix_timestamp` · `to_utc_timestamp` · `trunc` · `try_make_interval` · `try_make_timestamp` · `try_make_timestamp_ltz` · `try_make_timestamp_ntz` · `try_to_date` · `try_to_time` · `try_to_timestamp` · `unix_date` · `unix_micros` · `unix_millis` · `unix_nanos` · `unix_seconds` · `unix_timestamp` · `weekday` · `weekofyear` · `window` · `window_time` · `year`

Notes: format patterns are Java `DateTimeFormatter` style (`yyyy-MM-dd HH:mm:ss`) · `dayofweek` is 1 = Sunday, `weekday` is 0 = Monday ·
`TIME` and interval values are cast to `string` below only to display them. [Reference](https://spark.apache.org/docs/latest/api/sql/datetime-functions/)

In [0]:
t = spark.createDataFrame([(dt.date(2026, 11, 18), dt.datetime(2026, 11, 18, 13, 45, 30))], "d date, ts timestamp")

show_examples([
    # current values and arithmetic
    "F.current_date()", "F.current_timestamp()", "F.expr('current_timezone()')",
    "F.date_add('d', 10)", "F.date_sub('d', 10)", "F.add_months('d', 3)",
    "F.datediff(F.lit(dt.date(2026, 12, 24)), 'd')", "F.expr(\"date_diff(DATE'2026-12-24', d)\")",
    "F.months_between(F.lit(dt.date(2027, 2, 18)), 'd')", "F.last_day('d')", "F.next_day('d', 'Mon')",
    "F.expr('ts + INTERVAL 90 MINUTES')",
    # truncation and fields
    "F.date_trunc('month', 'ts')", "F.trunc('d', 'year')",
    "F.year('d')", "F.quarter('d')", "F.month('d')", "F.dayofmonth('d')", "F.dayofweek('d')", "F.expr('weekday(d)')",
    "F.dayofyear('d')", "F.weekofyear('d')", "F.hour('ts')", "F.minute('ts')", "F.second('ts')",
    "F.expr('dayname(d)')", "F.expr('monthname(d)')", "F.expr('extract(DOW FROM d)')", "F.expr(\"date_part('YEAR', ts)\")",
    # build, format, parse
    "F.make_date(F.lit(2026), F.lit(12), F.lit(24))",
    "F.expr('make_timestamp(2026, 11, 18, 13, 45, 30.5)')",
    "F.date_format('ts', 'EEE, dd MMM yyyy HH:mm')",
    "F.to_date(F.lit('18/11/2026'), 'dd/MM/yyyy')",
    "F.to_timestamp(F.lit('2026-11-18 13:45'), 'yyyy-MM-dd HH:mm')",
    "F.expr(\"try_to_timestamp('not a date')\")",
    # epoch and time zones
    "F.unix_timestamp('ts')", "F.from_unixtime(F.lit(0))", "F.timestamp_seconds(F.lit(1_700_000_000))",
    "F.expr('unix_millis(ts)')", "F.expr('unix_date(d)')",
    "F.from_utc_timestamp('ts', 'Europe/Warsaw')", "F.to_utc_timestamp('ts', 'Europe/Warsaw')",
    "F.expr(\"convert_timezone('UTC', 'America/New_York', ts)\")",
    # intervals and the TIME type (cast to string to display them)
    "F.expr('string(make_interval(1, 2, 0, 3))')",
    "F.expr('make_dt_interval(1, 12, 30, 0)')",
    "F.expr('string(make_ym_interval(1, 6))')",
    "F.expr('string(make_time(13, 45, 30.5))')",
    "F.expr(\"string(time_trunc('HOUR', TIME'13:45:30'))\")",
    "F.expr(\"string(to_time('13:45:30'))\")",
], t)

# Event-time windows: tumbling 7-day windows over the last logins
display(emp.where("last_login IS NOT NULL")
           .groupBy(F.window("last_login", "7 days").alias("w"))
           .agg(F.count("*").alias("logins"))
           .select("w.start", "w.end", "logins").orderBy("start"))

## 10. Generator Functions (9)

Functions that return **several rows per input row**: `explode` / `posexplode` (array or map → rows, `pos` = 0-based index), the
`_outer` variants keep rows whose collection is empty or NULL, `inline` explodes an array of structs into columns, `stack(n, …)`
un-pivots. `collations()` and `sql_keywords()` are table-valued: they list the available collations and SQL keywords.

**Functions:** `collations` · `explode` · `explode_outer` · `inline` · `inline_outer` · `posexplode` · `posexplode_outer` · `sql_keywords` · `stack`

[Reference](https://spark.apache.org/docs/latest/api/sql/generator-functions/)

In [0]:
display(emp.select("name", F.explode("tags").alias("tag")))                       # Frank (empty array) disappears
display(emp.select("name", F.explode_outer("tags").alias("tag")))                 # ... and is kept here with NULL
display(emp.select("name", F.posexplode("scores").alias("pos", "score")))
display(emp.select("name", F.posexplode_outer("props").alias("pos", "key", "value")))
display(spark.range(1).select(F.inline(F.array(F.struct(F.lit("a").alias("k"), F.lit(1).alias("v")),
                                                F.struct(F.lit("b").alias("k"), F.lit(2).alias("v"))))))
display(emp.selectExpr("name", "stack(2, 'salary', double(salary), 'bonus', bonus) AS (metric, amount)"))
display(spark.sql("SELECT * FROM sql_keywords() LIMIT 5"))
display(spark.sql("SELECT * FROM collations() LIMIT 5"))

## 11. Hash Functions (7)

`crc32` (BIGINT), `hash` (32-bit Murmur3, seed 42, any number of arguments of any type), `xxhash64` (64-bit, seed 42), `md5`,
`sha` / `sha1` and `sha2(expr, bitLength)` (224 / 256 / 384 / 512; 0 = 256) which return hex strings of a binary input.

**Functions:** `crc32` · `hash` · `md5` · `sha` · `sha1` · `sha2` · `xxhash64`

Use `hash` / `xxhash64` for bucketing and deterministic keys (collisions possible), `sha2(…, 256)` for change detection — hash the
whole row as one value (`to_json(struct(*))`), because NULL arguments are skipped. [Reference](https://spark.apache.org/docs/latest/api/sql/hash-functions/)

In [0]:
show_examples([
    "F.crc32(F.lit('Spark'))",
    "F.hash(F.lit('Spark'), F.array(F.lit(123)), F.lit(2))",
    "F.xxhash64(F.lit('Spark'), F.array(F.lit(123)), F.lit(2))",
    "F.md5(F.lit('Spark'))",
    "F.expr(\"sha('Spark')\")",
    "F.sha1(F.lit('Spark'))",
    "F.sha2(F.lit('Spark'), 256)",
    "F.sha2(F.lit('Spark'), 512)",
    "F.sha2(F.to_json(F.struct('emp_id', 'name', 'bonus')), 256)",     # NULL-safe row fingerprint
], emp)

## 12. Json Functions (7)

Work with JSON **strings**: parse into a typed struct / array / map (`from_json`, schema in DDL), extract one value by JSONPath
(`get_json_object`, `json_tuple` for several top-level fields at once), inspect (`json_array_length`, `json_object_keys`,
`schema_of_json`) and serialize (`to_json`). For semi-structured data that you query often, see **Variant functions** (section 24).

**Functions:** `from_json` · `get_json_object` · `json_array_length` · `json_object_keys` · `json_tuple` · `schema_of_json` · `to_json`

[Reference](https://spark.apache.org/docs/latest/api/sql/json-functions/)

In [0]:
js = spark.createDataFrame([('{"id": 7, "name": "Alice", "skills": ["sql", "spark"], "addr": {"city": "Wroclaw"}}',)], "j string")

show_examples([
    "F.from_json('j', 'id INT, name STRING, skills ARRAY<STRING>, addr STRUCT<city: STRING>')",
    "F.from_json('j', 'id INT, name STRING, skills ARRAY<STRING>, addr STRUCT<city: STRING>').addr.city",
    "F.get_json_object('j', '$.skills[1]')",
    "F.get_json_object('j', '$.addr.city')",
    "F.json_array_length(F.get_json_object('j', '$.skills'))",
    "F.json_object_keys('j')",
    "F.schema_of_json(F.lit('{\"a\": 1, \"b\": [true]}'))",
    "F.to_json(F.struct(F.lit(1).alias('a'), F.array(F.lit('x')).alias('b')))",
    "F.to_json(F.create_map(F.lit('k'), F.lit('v')))",
], js)

# json_tuple is a generator: several top-level fields as columns
display(js.select(F.json_tuple("j", "id", "name").alias("id", "name")))

## 13. Map Functions (9)

Build (`map`, `map_from_arrays`, `map_from_entries`, `str_to_map`), combine (`map_concat`), test (`map_contains_key`) and take
apart (`map_keys`, `map_values`, `map_entries`) `MAP` values. In PySpark `map(...)` is `F.create_map`. Lookup by key:
`col['key']` or `element_at(m, key)`.

**Functions:** `map` · `map_concat` · `map_contains_key` · `map_entries` · `map_from_arrays` · `map_from_entries` · `map_keys` · `map_values` · `str_to_map`

[Reference](https://spark.apache.org/docs/latest/api/sql/map-functions/)

In [0]:
show_examples([
    "F.create_map(F.lit('a'), F.lit(1), F.lit('b'), F.lit(2))",
    "F.map_concat('props', F.create_map(F.lit('team'), F.lit('core')))",
    "F.map_contains_key('props', 'site')",
    "F.map_entries('props')",
    "F.map_from_arrays(F.array(F.lit('x'), F.lit('y')), F.array(F.lit(1), F.lit(2)))",
    "F.map_from_entries(F.array(F.struct(F.lit('k'), F.lit(10))))",
    "F.map_keys('props')",
    "F.map_values('props')",
    "F.str_to_map(F.lit('a:1,b:2'), F.lit(','), F.lit(':'))",
    "F.col('props')['level']",
    "F.element_at('props', F.lit('site'))",
], emp)

## 14. Math Functions (68)

Arithmetic operators (`+ - * / %`, `div` = integral division), rounding (`round` = HALF_UP, `bround` = HALF_EVEN, `ceil`, `floor`,
`rint`), powers and logs, trigonometry, number bases (`bin`, `hex`, `unhex`, `conv`), `greatest` / `least`, random numbers
(`rand`, `randn`, 🆕 `uniform`, `random`), bucketing (`width_bucket`) and the ANSI-safe `try_add` / `try_subtract` / `try_multiply`
/ `try_divide` / `try_mod` that return NULL instead of failing on overflow or division by zero.

**Functions:** `%` · `*` · `+` · `-` · `/` · `abs` · `acos` · `acosh` · `asin` · `asinh` · `atan` · `atan2` · `atanh` · `bin` · `bround` · `cbrt` · `ceil` · `ceiling` · `conv` · `cos` · `cosh` · `cot` · `csc` · `degrees` · `div` · `e` · `exp` · `expm1` · `factorial` · `floor` · `greatest` · `hex` · `hypot` · `least` · `ln` · `log` · `log10` · `log1p` · `log2` · `mod` · `negative` · `pi` · `pmod` · `positive` · `pow` · `power` · `radians` · `rand` · `randn` · `random` · `rint` · `round` · `sec` · `sign` · `signum` · `sin` · `sinh` · `sqrt` · `tan` · `tanh` · `try_add` · `try_divide` · `try_mod` · `try_multiply` · `try_subtract` · `unhex` · `uniform` · `width_bucket`

Note: `%` / `mod` keep the sign of the dividend (`-7 % 3 = -1`), `pmod` is always non-negative (`2`). [Reference](https://spark.apache.org/docs/latest/api/sql/math-functions/)

In [0]:
num = spark.createDataFrame([(-7, 3, 2.5, 16.0)], "a int, b int, x double, y double")

show_examples([
    "F.col('a') + F.col('b')", "F.col('a') / F.col('b')", "F.expr('a div b')", "F.col('a') % F.col('b')",
    "F.pmod('a', 'b')", "F.expr('mod(a, b)')", "F.negative('b')", "F.abs('a')",
    "F.round('x')", "F.bround('x')", "F.round(F.lit(3.14159), 2)", "F.ceil('x')", "F.floor('x')", "F.rint('x')",
    "F.sqrt('y')", "F.cbrt(F.lit(27))", "F.pow('y', 0.5)", "F.exp(F.lit(1))", "F.expm1(F.lit(1e-10))",
    "F.ln(F.e())", "F.log(2.0, 'y')", "F.log10(F.lit(1000))", "F.log2('y')", "F.log1p(F.lit(0.0))",
    "F.pi()", "F.e()", "F.degrees(F.pi())", "F.radians(F.lit(180))",
    "F.sin(F.pi() / 2)", "F.cos(F.lit(0))", "F.tan(F.lit(0))", "F.atan2(F.lit(1), F.lit(1))", "F.hypot(F.lit(3), F.lit(4))",
    "F.expr('cot(1)')", "F.expr('sec(0)')", "F.expr('csc(pi() / 2)')", "F.asinh(F.lit(0))",
    "F.sign('a')", "F.factorial(F.lit(5))", "F.greatest('a', 'b', F.lit(0))", "F.least('a', 'b', F.lit(0))",
    "F.bin(F.lit(10))", "F.hex(F.lit(255))", "F.unhex(F.lit('FF'))", "F.conv(F.lit('ff'), 16, 10)",
    "F.width_bucket(F.lit(5.3), F.lit(0.2), F.lit(10.6), F.lit(5))",
    "F.try_divide(F.lit(1), F.lit(0))", "F.try_add(F.lit(2147483647), F.lit(1))", "F.try_multiply(F.lit(2), F.lit(3))",
    "F.try_subtract(F.lit(5), F.lit(7))", "F.expr('try_mod(7, 0)')",
    "F.rand(42)", "F.randn(42)", "F.expr('uniform(1, 10, 42)')", "F.expr('random(42)')",
], num)

## 15. Misc Functions (26)

Session and environment information (`current_user`, `current_catalog`, `current_schema` / `current_database`, `version`,
`typeof`), identifiers (`uuid`, `monotonically_increasing_id` — unique but **not** consecutive, `spark_partition_id`), file
lineage (`input_file_name`, `input_file_block_*`, empty when the data does not come from files), encryption (`aes_encrypt` /
`aes_decrypt` / `try_aes_decrypt`), data-quality guards (`assert_true`, `raise_error`), bitmap helpers used with the
`bitmap_*_agg` aggregates, and JVM reflection (`reflect`, `java_method`, `try_reflect`).

**Functions:** `aes_decrypt` · `aes_encrypt` · `assert_true` · `bitmap_bit_position` · `bitmap_bucket_number` · `bitmap_count` · `current_catalog` · `current_database` · `current_path` · `current_schema` · `current_user` · `input_file_block_length` · `input_file_block_start` · `input_file_name` · `java_method` · `monotonically_increasing_id` · `raise_error` · `reflect` · `session_user` · `spark_partition_id` · `try_aes_decrypt` · `try_reflect` · `typeof` · `user` · `uuid` · `version`

🟠 On Databricks with Unity Catalog `input_file_name` / `input_file_block_*` are not supported — use the `_metadata.file_path` (`_metadata.file_block_start`, …) column of a file source.
[Reference](https://spark.apache.org/docs/latest/api/sql/misc-functions/)

In [0]:
show_examples([
    "F.current_user()", "F.expr('session_user()')", "F.expr('user()')",
    "F.current_catalog()", "F.current_database()", "F.current_schema()",
    "F.version()", "F.expr('typeof(map(1, array(2.5)))')", "F.expr('uuid()')",
    "F.expr(\"cast(aes_decrypt(aes_encrypt('Spark', '0000111122223333'), '0000111122223333') AS STRING)\")",
    "F.expr(\"try_aes_decrypt(unhex('00'), '0000111122223333')\")",       # NULL instead of an error
    "F.assert_true(F.lit(1) < F.lit(2))",                                  # NULL when the condition holds
    "F.expr('bitmap_bit_position(1)')", "F.expr('bitmap_bucket_number(123456)')",
])

# File lineage: on Unity Catalog input_file_name() is blocked -> read the hidden _metadata column of a file source instead
# (any file-based read exposes it, e.g. spark.read.format("csv").load(path).select("_metadata.file_path"))

# identifiers that depend on rows and partitions
display(spark.range(0, 8, numPartitions=3)
             .select("id", F.monotonically_increasing_id().alias("mono_id"), F.spark_partition_id().alias("partition")))

# raise_error stops the query with your message (caught here to keep the notebook running)
try:
    spark.range(1).select(F.raise_error(F.lit("data-quality check failed"))).collect()
except Exception as ex:
    print("raise_error ->", str(ex).splitlines()[0][:120])

## 16. Predicate Functions (23)

Boolean tests: comparisons (`=`, `==`, `!=`, `<>`, `<`, `<=`, `>`, `>=`), the **NULL-safe equality** `<=>` / `equal_null`
(`NULL <=> NULL` is true), logic (`and`, `or`, `not` / `!`), membership (`in`), NULL / NaN checks and pattern matching
(`like`, case-insensitive `ilike`, regular expressions `rlike` / `regexp` / `regexp_like`).
PySpark spells them `==`, `!=`, `&`, `|`, `~`, `Column.eqNullSafe`, `isin`, `isNull`, `like`, `ilike`, `rlike`.

**Functions:** `!` · `!=` · `<` · `<=` · `<=>` · `<>` · `=` · `==` · `>` · `>=` · `and` · `equal_null` · `ilike` · `in` · `isnan` · `isnotnull` · `isnull` · `like` · `not` · `or` · `regexp` · `regexp_like` · `rlike`

Note: with a NULL operand `=` returns NULL (not false) — filters then drop the row. [Reference](https://spark.apache.org/docs/latest/api/sql/predicate-functions/)

In [0]:
pred = spark.createDataFrame([(1, 2, None, "Spark SQL", float("nan"))], "a int, b int, n int, s string, d double")

show_examples([
    "F.col('a') == F.col('b')", "F.col('a') != F.col('b')", "F.col('a') < F.col('b')", "F.col('a') >= F.col('b')",
    "F.col('n') == F.lit(None)",                      # NULL, not false
    "F.col('n').eqNullSafe(F.lit(None))",             # <=>
    "F.equal_null('n', F.lit(None))",
    "(F.col('a') < 2) & (F.col('b') > 1)", "(F.col('a') > 5) | F.col('n').isNull()", "~(F.col('a') == 1)",
    "F.col('a').isin(1, 3, 5)", "F.expr('a IN (2, 4)')",
    "F.col('n').isNull()", "F.col('n').isNotNull()", "F.isnan('d')",
    "F.col('s').like('Spark%')", "F.col('s').ilike('spark%')", "F.col('s').rlike('^S.*L$')",
    "F.regexp_like('s', F.lit('[0-9]'))", "F.expr(\"regexp(s, 'SQL')\")",
    "F.expr('a <> b')", "F.expr('NOT (a = b)')",
], pred)

## 17. Protobuf Functions (2)

Serialize a struct to **Protobuf binary** (`to_protobuf`) and back (`from_protobuf`). Both need the message definition: a compiled
descriptor set (a file created by `protoc --descriptor_set_out --include_imports`, or its bytes) plus the message name.
The cell below builds a tiny descriptor in Python, so no `.proto` file or `protoc` is needed.

**Functions:** `from_protobuf` · `to_protobuf`

Since Spark 4.0 in SQL. Note: the schema must match the data, otherwise the result is undefined. [Reference](https://spark.apache.org/docs/latest/api/sql/protobuf-functions/)

In [0]:
from google.protobuf import descriptor_pb2
from pyspark.sql.protobuf.functions import to_protobuf, from_protobuf

# message demo.Person { string name = 1; int32 id = 2; } — built in code instead of a .proto file
fdp = descriptor_pb2.FileDescriptorProto(name="person.proto", package="demo", syntax="proto3")
person = fdp.message_type.add(name="Person")
person.field.add(name="name", number=1, type=descriptor_pb2.FieldDescriptorProto.TYPE_STRING,
                 label=descriptor_pb2.FieldDescriptorProto.LABEL_OPTIONAL)
person.field.add(name="id", number=2, type=descriptor_pb2.FieldDescriptorProto.TYPE_INT32,
                 label=descriptor_pb2.FieldDescriptorProto.LABEL_OPTIONAL)
desc_bytes = descriptor_pb2.FileDescriptorSet(file=[fdp]).SerializeToString()

pb = emp.select(to_protobuf(F.struct("name", F.col("emp_id").alias("id")), "demo.Person",
                            binaryDescriptorSet=desc_bytes).alias("proto_bytes"))
display(pb.select(F.hex("proto_bytes").alias("proto_hex"),
                  from_protobuf("proto_bytes", "demo.Person", binaryDescriptorSet=desc_bytes).alias("decoded")))

## 18. Sketch Functions (40)

Read and combine the **sketches** built by the sketch aggregates of section 1: HyperLogLog (`hll_*`, distinct counts), Theta
(`theta_*`, distinct counts with set operations: union / intersection / difference), KLL (`kll_sketch_*`, quantiles and ranks),
Tuple sketches (`tuple_*`, distinct keys carrying a summary value) and `approx_top_k_estimate` (most frequent items).
Sketches are small binary values: store them per day / partition and merge them later instead of rescanning the raw data.

**Functions:** `approx_top_k_estimate` · `hll_sketch_estimate` · `hll_union` · `kll_sketch_get_n_bigint` · `kll_sketch_get_n_double` · `kll_sketch_get_n_float` · `kll_sketch_get_quantile_bigint` · `kll_sketch_get_quantile_double` · `kll_sketch_get_quantile_float` · `kll_sketch_get_rank_bigint` · `kll_sketch_get_rank_double` · `kll_sketch_get_rank_float` · `kll_sketch_merge_bigint` · `kll_sketch_merge_double` · `kll_sketch_merge_float` · `kll_sketch_to_string_bigint` · `kll_sketch_to_string_double` · `kll_sketch_to_string_float` · `theta_difference` · `theta_intersection` · `theta_sketch_estimate` · `theta_union` · `tuple_difference_double` · `tuple_difference_integer` · `tuple_difference_theta_double` · `tuple_difference_theta_integer` · `tuple_intersection_double` · `tuple_intersection_integer` · `tuple_intersection_theta_double` · `tuple_intersection_theta_integer` · `tuple_sketch_estimate_double` · `tuple_sketch_estimate_integer` · `tuple_sketch_summary_double` · `tuple_sketch_summary_integer` · `tuple_sketch_theta_double` · `tuple_sketch_theta_integer` · `tuple_union_double` · `tuple_union_integer` · `tuple_union_theta_double` · `tuple_union_theta_integer`

[Reference](https://spark.apache.org/docs/latest/api/sql/sketch-functions/)

In [0]:
# One sketch per department, then estimates and set operations between two of them
sk = spark.sql("""
    SELECT dept,
           hll_sketch_agg(name)                          AS hll,
           theta_sketch_agg(name)                        AS theta,
           kll_sketch_agg_bigint(CAST(salary AS BIGINT)) AS kll
    FROM emp GROUP BY dept""")
sk.createOrReplaceTempView("sketches")

display(spark.sql("""
    SELECT dept,
           hll_sketch_estimate(hll)                    AS hll_distinct_names,
           theta_sketch_estimate(theta)                AS theta_distinct_names,
           kll_sketch_get_n_bigint(kll)                AS kll_n,
           kll_sketch_get_quantile_bigint(kll, 0.5)    AS kll_median_salary,
           kll_sketch_get_rank_bigint(kll, 150000)     AS kll_rank_of_150k
    FROM sketches ORDER BY dept"""))

display(spark.sql("""
    SELECT hll_sketch_estimate(hll_union(e.hll, s.hll))                          AS hll_union_estimate,
           theta_sketch_estimate(theta_union(e.theta, s.theta))                  AS theta_union_estimate,
           theta_sketch_estimate(theta_intersection(e.theta, s.theta))           AS theta_intersection_estimate,
           theta_sketch_estimate(theta_difference(e.theta, s.theta))             AS theta_difference_estimate,
           kll_sketch_get_quantile_bigint(kll_sketch_merge_bigint(e.kll, s.kll), 0.5) AS merged_median
    FROM sketches e CROSS JOIN sketches s
    WHERE e.dept = 'Engineering' AND s.dept = 'Sales'"""))

display(spark.sql("""
    SELECT approx_top_k_estimate(approx_top_k_accumulate(dept), 2) AS top_2_departments FROM emp"""))

## 19. St Functions (5)

🆕 Spark 4.1 geospatial basics for the new `GEOMETRY` / `GEOGRAPHY` types: parse WKB (`st_geomfromwkb`, `st_geogfromwkb`), write
WKB (`st_asbinary`, `'NDR'` little-endian or `'XDR'` big-endian) and read / set the spatial reference id (`st_srid`, `st_setsrid`).
GEOGRAPHY values default to SRID 4326 (WGS 84), GEOMETRY values to 0. 🟠 Databricks has a much larger `ST_*` library.

**Functions:** `st_asbinary` · `st_geogfromwkb` · `st_geomfromwkb` · `st_setsrid` · `st_srid`

Values are shown as hex WKB or SRID numbers. [Reference](https://spark.apache.org/docs/latest/api/sql/st-functions/)

In [0]:
WKB = "X'0101000000000000000000F03F0000000000000040'"          # POINT(1 2) in WKB
show_examples([
    f"F.expr(\"hex(st_asbinary(st_geogfromwkb({WKB})))\")",
    f"F.expr(\"hex(st_asbinary(st_geomfromwkb({WKB}), 'XDR'))\")",
    f"F.expr(\"st_srid(st_geogfromwkb({WKB}))\")",
    f"F.expr(\"st_srid(st_geomfromwkb({WKB}))\")",
    f"F.expr(\"st_srid(st_geomfromwkb({WKB}, 4326))\")",
    f"F.expr(\"st_srid(st_setsrid(st_geomfromwkb({WKB}), 3857))\")",
])

## 20. String Functions (74)

The largest everyday group: length and case, trimming and padding, substrings (`substring` / `substr` are **1-based**, `left`,
`right`, `split_part`, `substring_index`), search (`instr`, `locate`, `position`, `contains`, `startswith`, `endswith`,
`find_in_set`), replace (`replace`, `translate`, `overlay`), regular expressions (`regexp_extract[_all]`, `regexp_replace`,
`regexp_count`, `regexp_substr`, `regexp_instr`), formatting (`format_number`, `format_string` / `printf`, `to_char`,
`to_number`), encodings (`base64`, `encode` / `decode`, `to_binary`), fuzzy matching (`levenshtein`, `soundex`,
`jaro_winkler_similarity`), masking (`mask`), 🆕 collations (`collate`, `collation`) and UTF-8 validation.

**Functions:** `ascii` · `base64` · `bit_length` · `btrim` · `char` · `char_length` · `character_length` · `chr` · `collate` · `collation` · `concat_ws` · `contains` · `decode` · `elt` · `encode` · `endswith` · `find_in_set` · `format_number` · `format_string` · `from_base32` · `initcap` · `instr` · `is_valid_utf8` · `jaro_winkler_similarity` · `lcase` · `left` · `len` · `length` · `levenshtein` · `locate` · `lower` · `lpad` · `ltrim` · `luhn_check` · `make_valid_utf8` · `mask` · `octet_length` · `overlay` · `position` · `printf` · `quote` · `randstr` · `regexp_count` · `regexp_extract` · `regexp_extract_all` · `regexp_instr` · `regexp_replace` · `regexp_substr` · `repeat` · `replace` · `right` · `rpad` · `rtrim` · `sentences` · `soundex` · `space` · `split` · `split_part` · `startswith` · `substr` · `substring` · `substring_index` · `to_base32` · `to_binary` · `to_char` · `to_number` · `to_varchar` · `translate` · `trim` · `try_to_binary` · `try_to_number` · `try_validate_utf8` · `ucase` · `unbase64` · `upper` · `validate_utf8` · `||`

Note: regex patterns are Java regular expressions; in a SQL string literal a backslash must be doubled. [Reference](https://spark.apache.org/docs/latest/api/sql/string-functions/)

In [0]:
s = spark.createDataFrame([("  Spark SQL  ", "Bartosz Malec", "order-2026-11-18-PL", "a,b,c")], "pad string, name string, code string, csv string")

show_examples([
    # length, case, trim, pad
    "F.length('name')", "F.char_length('name')", "F.bit_length('name')", "F.octet_length(F.lit('zażółć'))",
    "F.lower('name')", "F.upper('name')", "F.initcap(F.lit('hello spark sql'))",
    "F.trim('pad')", "F.ltrim('pad')", "F.rtrim('pad')", "F.btrim(F.lit('xxSparkxx'), F.lit('x'))",
    "F.lpad(F.lit('42'), 6, '0')", "F.rpad(F.lit('42'), 6, '*')", "F.repeat(F.lit('ab'), 3)", "F.expr('space(3)')",
    # concat, substrings, split
    "F.concat_ws(' | ', 'name', 'code')", "F.expr(\"'Spark' || ' ' || 'SQL'\")",
    "F.substring('name', 1, 7)", "F.left('name', F.lit(3))", "F.right('name', F.lit(5))",
    "F.substring_index('code', '-', 2)", "F.split('csv', ',')", "F.split_part('code', F.lit('-'), F.lit(2))",
    "F.expr(\"elt(2, 'a', 'b', 'c')\")",
    # search and replace
    "F.instr('name', 'Malec')", "F.locate('a', 'name', 3)", "F.position(F.lit('Malec'), 'name')",
    "F.contains('name', F.lit('Mal'))", "F.startswith('code', F.lit('order'))", "F.endswith('code', F.lit('PL'))",
    "F.find_in_set(F.lit('b'), 'csv')",
    "F.replace('code', F.lit('-'), F.lit('/'))", "F.translate('code', '-', '_')", "F.overlay('name', F.lit('X'), 1, 7)",
    "F.reverse('name')",
    # regular expressions
    "F.regexp_extract('code', r'(\\d{4})-(\\d{2})', 2)",
    "F.regexp_extract_all('code', F.lit(r'(\\d+)'), 1)",
    "F.regexp_replace('code', r'\\d', '#')",
    "F.regexp_count('code', F.lit(r'\\d+'))",
    "F.regexp_substr('code', F.lit(r'\\d{4}'))",
    "F.regexp_instr('code', F.lit(r'\\d'))",
    # formatting and numbers <-> strings
    "F.format_number(F.lit(1234567.891), 2)", "F.format_string('%s earns %,d', F.lit('Bob'), F.lit(200000))",
    "F.printf(F.lit('%05d'), F.lit(42))", "F.to_char(F.lit(1234.5), F.lit('$9,999.00'))",
    "F.to_number(F.lit('$1,234.50'), F.lit('$9,999.99'))", "F.try_to_number(F.lit('abc'), F.lit('999'))",
    # encodings and characters
    "F.ascii(F.lit('A'))", "F.expr('chr(65)')", "F.base64(F.lit('Spark'))", "F.unbase64(F.lit('U3Bhcms='))",
    "F.encode(F.lit('Spark'), 'UTF-8')", "F.decode(F.encode(F.lit('Spark'), 'UTF-8'), 'UTF-8')",
    "F.to_binary(F.lit('537061726B'), F.lit('hex'))", "F.expr(\"is_valid_utf8('Spark')\")",
    # fuzzy matching, masking, misc
    "F.levenshtein(F.lit('kitten'), F.lit('sitting'))", "F.soundex(F.lit('Robert'))",
    "F.expr(\"jaro_winkler_similarity('Malec', 'Malek')\")",
    "F.mask(F.lit('AbCD123-@$#'))", "F.expr(\"luhn_check('79927398713')\")",
    "F.sentences(F.lit('Hi there! How are you?'))", "F.expr(\"quote('Don''t')\")",
    "F.expr(\"collation('abc' COLLATE UNICODE_CI)\")",
    "F.expr(\"'Spark' = 'SPARK' COLLATE UTF8_LCASE\")",
    "F.expr('length(randstr(8, 0))')",
], s)

## 21. Struct Functions (2)

Build a `STRUCT`: `struct(col1, col2, …)` (field names come from the columns, otherwise `col1`, `col2`, …) or
`named_struct('name1', val1, …)`. In PySpark also: read a field with `col.fieldName` / `col['field']`, change one with
`withField`, remove with `dropFields`, and flatten with `select('s.*')`.

**Functions:** `named_struct` · `struct`

[Reference](https://spark.apache.org/docs/latest/api/sql/struct-functions/)

In [0]:
show_examples([
    "F.struct(F.lit(1), F.lit(2), F.lit(3))",
    "F.struct('emp_id', 'name')",
    "F.named_struct(F.lit('a'), F.lit(1), F.lit('b'), F.lit(2))",
    "F.struct('emp_id', 'name').getField('name')",
    "F.struct('emp_id', 'name').withField('name', F.upper('name'))",
    "F.struct('emp_id', 'name', 'dept').dropFields('dept')",
], emp)

display(emp.select(F.struct("emp_id", "name", "dept").alias("s")).select("s.*"))     # struct -> columns

## 22. Table Functions (2)

Functions used in the `FROM` clause that **return a table**. `range(start, end, step, numParts)` generates a BIGINT column `id`
(PySpark: `spark.range`). 🆕 `python_worker_logs()` (Spark 4.1) returns the logs collected from Python workers once
`spark.sql.pyspark.worker.logging.enabled` is on.

**Functions:** `python_worker_logs` · `range`

[Reference](https://spark.apache.org/docs/latest/api/sql/table-functions/)

In [0]:
display(spark.sql("SELECT * FROM range(0, 10, 3)"))          # SQL table function
display(spark.range(0, 10, 3))                               # the PySpark equivalent
display(spark.sql("SELECT id, id * id AS square FROM range(5)"))

## 23. Url Functions (5)

`parse_url(url, part[, key])` extracts `HOST`, `PATH`, `QUERY`, `REF`, `PROTOCOL`, `AUTHORITY`, `FILE`, `USERINFO` or one query
parameter; `url_encode` / `url_decode` do `application/x-www-form-urlencoded` encoding. The `try_` versions return NULL for
malformed input instead of failing.

**Functions:** `parse_url` · `try_parse_url` · `try_url_decode` · `url_decode` · `url_encode`

[Reference](https://spark.apache.org/docs/latest/api/sql/url-functions/)

In [0]:
u = spark.createDataFrame([("https://user@spark.apache.org:443/docs/latest/api/sql/?lang=en&q=hash#url",)], "url string")

show_examples([
    "F.parse_url('url', F.lit('HOST'))",
    "F.parse_url('url', F.lit('PATH'))",
    "F.parse_url('url', F.lit('QUERY'))",
    "F.parse_url('url', F.lit('QUERY'), F.lit('q'))",
    "F.parse_url('url', F.lit('REF'))",
    "F.parse_url('url', F.lit('PROTOCOL'))",
    "F.parse_url('url', F.lit('AUTHORITY'))",
    "F.parse_url('url', F.lit('USERINFO'))",
    "F.url_encode(F.lit('a b&c=ą'))",
    "F.url_decode(F.lit('a+b%26c%3D%C4%85'))",
    "F.expr(\"try_url_decode('%E0%A4%A')\")",
    "F.expr(\"try_parse_url('inva lid://x', 'HOST')\")",
], u)

## 24. Variant Functions (19)

🆕 The `VARIANT` type (Spark 4.0) stores semi-structured data in a binary form that is parsed **once** and queried fast:
`parse_json` / `try_parse_json` to create it, `variant_get` / `try_variant_get` (path + target type) to read it, `schema_of_variant`
(one value) and `schema_of_variant_agg` (many rows) to discover its shape, `variant_explode` to turn arrays / objects into rows,
and functions that edit a variant in place (`variant_insert`, `variant_set`, `variant_delete`, `variant_array_append`, …).

**Functions:** `is_valid_variant` · `is_variant_null` · `parse_json` · `schema_of_variant` · `schema_of_variant_agg` · `to_variant_object` · `try_parse_json` · `try_variant_array_append` · `try_variant_get` · `try_variant_insert` · `try_variant_set` · `variant_array_append` · `variant_delete` · `variant_explode` · `variant_explode_outer` · `variant_get` · `variant_insert` · `variant_set` · `variant_strip_nulls`

Variant values are shown with `to_json` below. [Reference](https://spark.apache.org/docs/latest/api/sql/variant-functions/)

In [0]:
v = spark.createDataFrame([('{"id": 7, "name": "Alice", "skills": ["sql", "spark"], "manager": null}',)], "j string") \
         .select(F.parse_json("j").alias("v"))

show_examples([
    "F.to_json('v')",
    "F.variant_get('v', '$.id', 'int')",
    "F.variant_get('v', '$.skills[1]', 'string')",
    "F.try_variant_get('v', '$.name', 'int')",                  # NULL: cannot cast 'Alice' to int
    "F.schema_of_variant('v')",
    "F.is_variant_null(F.variant_get('v', '$.manager', 'variant'))",
    "F.to_json(F.try_parse_json(F.lit('{broken')))",
    "F.expr(\"to_json(variant_insert(v, '$.city', 'Wroclaw'))\")",
    "F.expr(\"to_json(variant_set(v, '$.id', 8))\")",
    "F.expr(\"to_json(variant_delete(v, '$.skills'))\")",
    "F.expr(\"to_json(variant_array_append(v, '$.skills', 'python'))\")",
    "F.expr('to_json(variant_strip_nulls(v))')",
    "F.expr(\"to_json(to_variant_object(named_struct('a', 1, 'b', array(2, 3))))\")",
    "F.expr('is_valid_variant(v)')",
], v)

v.createOrReplaceTempView("variants")
display(spark.sql("SELECT key, to_json(value) AS value FROM variants, LATERAL variant_explode(v)"))
display(emp.select(F.parse_json(F.to_json(F.struct("emp_id", "name", "bonus"))).alias("v"))
           .agg(F.schema_of_variant_agg("v").alias("schema_of_variant_agg")))

## 25. Vector Functions (7)

🆕 Spark 4.2 functions for **embeddings** stored as `ARRAY<FLOAT>`: similarity / distance between two vectors
(`vector_cosine_similarity`, `vector_inner_product`, `vector_l2_distance`), the Lp norm and normalization (`vector_norm`,
`vector_normalize`; degree 1, 2 = default or infinity) and the element-wise aggregates `vector_sum` / `vector_avg`.
All vectors must have the same dimension.

**Functions:** `vector_avg` · `vector_cosine_similarity` · `vector_inner_product` · `vector_l2_distance` · `vector_norm` · `vector_normalize` · `vector_sum`

[Reference](https://spark.apache.org/docs/latest/api/sql/vector-functions/)

In [0]:
show_examples([
    "F.expr('vector_cosine_similarity(array(1.0F, 2.0F, 3.0F), array(4.0F, 5.0F, 6.0F))')",
    "F.expr('vector_inner_product(array(1.0F, 2.0F, 3.0F), array(4.0F, 5.0F, 6.0F))')",
    "F.expr('vector_l2_distance(array(1.0F, 2.0F, 3.0F), array(4.0F, 5.0F, 6.0F))')",
    "F.expr('vector_norm(array(3.0F, 4.0F), 2.0F)')",
    "F.expr('vector_norm(array(3.0F, 4.0F), 1.0F)')",
    "F.expr(\"vector_norm(array(3.0F, 4.0F), float('inf'))\")",
    "F.expr('vector_normalize(array(3.0F, 4.0F), 2.0F)')",
])

# nearest neighbour by cosine similarity + the two vector aggregates
docs = spark.createDataFrame([("a", [1.0, 0.0]), ("b", [0.7, 0.7]), ("c", [0.0, 1.0])], "doc string, emb array<float>")
query = "array(0.9F, 0.1F)"
display(docs.withColumn("similarity", F.expr(f"vector_cosine_similarity(emb, {query})")).orderBy(F.desc("similarity")))
display(docs.agg(F.expr("vector_sum(emb)").alias("vector_sum"), F.expr("vector_avg(emb)").alias("vector_avg")))

## 26. Window Functions (9)

Ranking and offset functions computed **over a window** — a partition of rows with an order — without collapsing the rows:
`row_number` (unique), `rank` (gaps after ties), `dense_rank` (no gaps), `percent_rank`, `cume_dist`, `ntile(n)`, `lag` / `lead`
(previous / next row) and `nth_value`. Every aggregate of section 1 can also run over a window (running totals, moving averages).

**Functions:** `cume_dist` · `dense_rank` · `lag` · `lead` · `nth_value` · `ntile` · `percent_rank` · `rank` · `row_number`

Note: an ordered window frame defaults to `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — set `rowsBetween` explicitly
when ties matter. [Reference](https://spark.apache.org/docs/latest/api/sql/window-functions/)

In [0]:
w = Window.partitionBy("dept").orderBy(F.desc("salary"))
display(emp.select(
    "dept", "name", "salary",
    F.row_number().over(w).alias("row_number"),
    F.rank().over(w).alias("rank"),
    F.dense_rank().over(w).alias("dense_rank"),
    F.round(F.percent_rank().over(w), 2).alias("percent_rank"),
    F.round(F.cume_dist().over(w), 2).alias("cume_dist"),
    F.ntile(2).over(w).alias("ntile_2"),
    F.lag("salary").over(w).alias("lag_salary"),
    F.lead("name").over(w).alias("lead_name"),
    F.nth_value("name", 2).over(w.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)).alias("nth_value_2"),
    F.sum("salary").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)).alias("running_total"),
).orderBy("dept", "row_number"))

## 27. Xml Functions (12)

🆕 `from_xml` / `to_xml` / `schema_of_xml` (Spark 4.0) convert between XML **strings** and structs, like their JSON and CSV
siblings. The `xpath*` family evaluates an XPath 1.0 expression on an XML string and returns a typed result: `xpath` (array of
strings), `xpath_string`, `xpath_int`, `xpath_long`, `xpath_short`, `xpath_float`, `xpath_double` / `xpath_number`,
`xpath_boolean`.

**Functions:** `from_xml` · `schema_of_xml` · `to_xml` · `xpath` · `xpath_boolean` · `xpath_double` · `xpath_float` · `xpath_int` · `xpath_long` · `xpath_number` · `xpath_short` · `xpath_string`

[Reference](https://spark.apache.org/docs/latest/api/sql/xml-functions/)

In [0]:
x = spark.createDataFrame([("<emp><id>7</id><name>Alice</name><salary>300000.5</salary>"
                            "<tags><tag>exec</tag><tag>founder</tag></tags></emp>",)], "xml string")

show_examples([
    "F.from_xml('xml', 'id INT, name STRING, salary DOUBLE')",
    "F.schema_of_xml(F.lit('<p><a>1</a><b>x</b></p>'))",
    "F.to_xml(F.struct(F.lit(7).alias('id'), F.lit('Alice').alias('name')))",
    "F.xpath('xml', F.lit('/emp/tags/tag/text()'))",
    "F.xpath_string('xml', F.lit('/emp/name'))",
    "F.xpath_int('xml', F.lit('/emp/id'))",
    "F.xpath_long('xml', F.lit('/emp/id'))",
    "F.xpath_short('xml', F.lit('/emp/id'))",
    "F.xpath_float('xml', F.lit('/emp/salary'))",
    "F.xpath_double('xml', F.lit('/emp/salary'))",
    "F.xpath_number('xml', F.lit('sum(/emp/id | /emp/salary)'))",
    "F.xpath_boolean('xml', F.lit('count(/emp/tags/tag) = 2'))",
], x)